# LESSON 5.1: Radon Transform Basics
## Image Reconstruction from Projections

In this lesson:
- What is the Radon Transform
- Line integrals and projections
- Mathematical definition of the Radon Transform
- Parallel-beam projection geometry
- Relationship to Computed Tomography (CT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon
from skimage.draw import disk, ellipse

## 1. Background: Image Reconstruction from Projections

**Johann Radon (1917)** proved that a 2-D function can be reconstructed from an infinite set of its projections.

### The Problem:
In **Computed Tomography (CT)**, X-ray beams pass through an object and sensors measure the total attenuation along each ray path. From these measurements (called **projections**), we want to reconstruct the internal structure of the object.

### Key Idea:
- An X-ray beam traveling through tissue is attenuated (weakened) by the materials it passes through
- The total attenuation along a ray is a **line integral** of the attenuation coefficient
- By collecting many such line integrals at different angles, we can reconstruct the 2-D distribution of attenuation coefficients

### Historical Context:
| Year | Event |
|------|-------|
| 1895 | Röntgen discovers X-rays |
| 1917 | Johann Radon develops the mathematical theory |
| 1963 | Allan Cormack develops reconstruction algorithms |
| 1971 | Godfrey Hounsfield builds the first CT scanner |
| 1979 | Cormack and Hounsfield receive the Nobel Prize in Medicine |

In [ ]:
# Visualize the concept of X-ray projection
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a simple phantom (cross-section of a body)
phantom = np.zeros((256, 256))
# Body outline (ellipse)
rr, cc = ellipse(128, 128, 100, 80)
valid = (rr >= 0) & (rr < 256) & (cc >= 0) & (cc < 256)
phantom[rr[valid], cc[valid]] = 0.5
# Bone
rr, cc = disk((128, 80), 20)
valid = (rr >= 0) & (rr < 256) & (cc >= 0) & (cc < 256)
phantom[rr[valid], cc[valid]] = 1.0
# Organ
rr, cc = disk((128, 170), 25)
valid = (rr >= 0) & (rr < 256) & (cc >= 0) & (cc < 256)
phantom[rr[valid], cc[valid]] = 0.8
# Small feature
rr, cc = disk((90, 128), 10)
valid = (rr >= 0) & (rr < 256) & (cc >= 0) & (cc < 256)
phantom[rr[valid], cc[valid]] = 0.9

# Show the phantom
axes[0].imshow(phantom, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Cross-Section (What We Want)', fontsize=12)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Show a single projection (vertical rays)
projection_0 = np.sum(phantom, axis=0)
axes[1].plot(projection_0, 'b-', linewidth=2)
axes[1].set_title('Single Projection at 0°', fontsize=12)
axes[1].set_xlabel('Detector Position')
axes[1].set_ylabel('Integrated Intensity')
axes[1].grid(True, alpha=0.3)

# Show projection at 90 degrees
projection_90 = np.sum(phantom, axis=1)
axes[2].plot(projection_90, 'r-', linewidth=2)
axes[2].set_title('Single Projection at 90°', fontsize=12)
axes[2].set_xlabel('Detector Position')
axes[2].set_ylabel('Integrated Intensity')
axes[2].grid(True, alpha=0.3)

plt.suptitle('CT Scanning Concept: Projections of a Cross-Section', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("A single projection gives us limited information.")
print("We need MANY projections at different angles to reconstruct the image!")

## 2. Line Integrals and X-Ray Attenuation

### Beer-Lambert Law

When an X-ray beam passes through a **homogeneous** material:

$$I = I_0 \, e^{-\mu t}$$

Where:
- $I_0$ = initial intensity
- $I$ = intensity after passing through material
- $\mu$ = linear attenuation coefficient (material property)
- $t$ = thickness of the material

### For Non-Homogeneous Material

When $\mu$ varies along the path, we use a **line integral**:

$$I = I_0 \, \exp\left(-\int_{\text{ray}} \mu(x, y) \, ds\right)$$

Taking the logarithm:

$$\boxed{\ln\frac{I_0}{I} = \int_{\text{ray}} \mu(x, y) \, ds}$$

The left side is **measurable** (we know $I_0$ and measure $I$).
The right side is the **line integral** of the unknown attenuation distribution.

In [ ]:
# Demonstrate Beer-Lambert law and line integrals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Beer-Lambert for homogeneous material
I0 = 1.0
mu_values = [0.1, 0.3, 0.5, 1.0]  # different attenuation coefficients
t = np.linspace(0, 10, 100)

for mu in mu_values:
    I = I0 * np.exp(-mu * t)
    axes[0].plot(t, I, linewidth=2, label=f'$\\mu$ = {mu}')

axes[0].set_title('Beer-Lambert Law: X-Ray Attenuation', fontsize=12)
axes[0].set_xlabel('Thickness t (cm)')
axes[0].set_ylabel('Intensity I/I₀')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Show a ray through non-homogeneous material
# Create a 1D attenuation profile
x = np.linspace(0, 10, 200)
mu_profile = np.zeros_like(x)
mu_profile[(x > 1) & (x < 3)] = 0.3   # soft tissue
mu_profile[(x > 3) & (x < 4)] = 1.0   # bone
mu_profile[(x > 4) & (x < 7)] = 0.3   # soft tissue
mu_profile[(x > 5) & (x < 5.5)] = 0.6  # organ
mu_profile[(x > 7) & (x < 9)] = 0.2   # soft tissue

axes[1].fill_between(x, 0, mu_profile, alpha=0.4, color='steelblue')
axes[1].plot(x, mu_profile, 'b-', linewidth=2)
axes[1].set_title('Non-Homogeneous Attenuation Along a Ray', fontsize=12)
axes[1].set_xlabel('Position along ray (cm)')
axes[1].set_ylabel('$\\mu$(x) (cm$^{-1}$)')
axes[1].grid(True, alpha=0.3)

# Annotate
line_integral = np.trapz(mu_profile, x)
axes[1].annotate(f'Line integral = {line_integral:.2f}', xy=(5, 0.8),
                fontsize=12, color='red', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('X-Ray Attenuation: From Measurement to Line Integral', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Mathematical Definition of the Radon Transform

The **Radon Transform** of a 2-D function $f(x, y)$ is defined as:

$$\boxed{g(\rho, \theta) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x, y) \, \delta(x\cos\theta + y\sin\theta - \rho) \, dx \, dy}$$

Where:
- $\rho$ = perpendicular distance from the origin to the ray
- $\theta$ = angle of the ray with respect to the x-axis
- $\delta(\cdot)$ = Dirac delta function (selects points on the line)
- $g(\rho, \theta)$ = the projection value (line integral along the ray)

### Geometry of a Ray

A line in the $(x, y)$ plane can be parameterized as:

$$x\cos\theta + y\sin\theta = \rho$$

This defines a line at angle $\theta$ and distance $\rho$ from the origin.

### Equivalent Form (Along the Ray)

Using the parameterization along the ray direction:

$$g(\rho, \theta) = \int_{-\infty}^{\infty} f(\rho\cos\theta - s\sin\theta, \; \rho\sin\theta + s\cos\theta) \, ds$$

where $s$ is the coordinate along the ray.

In [ ]:
# Visualize the geometry of the Radon Transform
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Draw the object (a circle)
theta_circle = np.linspace(0, 2*np.pi, 100)
ax.plot(2*np.cos(theta_circle), 2*np.sin(theta_circle), 'b-', linewidth=2, label='Object')
ax.fill(2*np.cos(theta_circle), 2*np.sin(theta_circle), alpha=0.1, color='blue')

# Draw a specific ray
theta_ray = np.radians(30)  # 30 degrees
rho = 1.0  # distance from origin

# Direction perpendicular to ray (normal)
nx, ny = np.cos(theta_ray), np.sin(theta_ray)
# Direction along ray
dx, dy = -np.sin(theta_ray), np.cos(theta_ray)

# Ray center point
cx, cy = rho * nx, rho * ny

# Draw the ray
s = np.linspace(-4, 4, 100)
ray_x = cx + s * dx
ray_y = cy + s * dy
ax.plot(ray_x, ray_y, 'r-', linewidth=2, label=f'Ray ($\\theta$={30}°, $\\rho$={rho})')

# Draw rho (perpendicular distance)
ax.plot([0, cx], [0, cy], 'g-', linewidth=2, label=f'$\\rho$ = {rho}')
ax.plot(cx, cy, 'go', markersize=10)

# Draw theta angle
angle_arc = np.linspace(0, theta_ray, 30)
ax.plot(0.5*np.cos(angle_arc), 0.5*np.sin(angle_arc), 'g-', linewidth=1.5)
ax.annotate('$\\theta$', xy=(0.4, 0.15), fontsize=16, color='green')

# Draw coordinate axes
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.plot(0, 0, 'ko', markersize=5)

# Annotations
ax.annotate('$\\rho$', xy=(0.3, 0.4), fontsize=16, color='green')
ax.annotate('Origin', xy=(0.1, -0.3), fontsize=10)
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', fontsize=14)

# Draw normal direction
ax.annotate('', xy=(cx + 0.5*nx, cy + 0.5*ny),
           xytext=(cx, cy),
           arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

ax.set_xlim([-4, 4])
ax.set_ylim([-4, 4])
ax.set_aspect('equal')
ax.legend(fontsize=11, loc='upper right')
ax.set_title('Radon Transform Geometry\n$x\\cos\\theta + y\\sin\\theta = \\rho$', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Parallel-Beam Projection Geometry

In **parallel-beam** CT, all rays at a given angle $\theta$ are parallel:

```
   X-ray Source (at infinity)
         |  |  |  |  |  |
         |  |  |  |  |  |   (parallel rays)
         v  v  v  v  v  v
    +-----------------------+
    |                       |
    |     Object f(x,y)     |
    |                       |
    +-----------------------+
         |  |  |  |  |  |
    [  Detector Array  ]
    (measures projections)
```

A **projection** $g(\rho, \theta_k)$ at angle $\theta_k$ consists of all line integrals for different $\rho$ values at that angle.

The collection of all projections for $\theta \in [0°, 180°)$ is called a **sinogram**.

In [ ]:
# Demonstrate parallel-beam projections at different angles
# Create the phantom
size = 256
phantom = np.zeros((size, size))
rr, cc = ellipse(128, 128, 100, 80)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom[rr[valid], cc[valid]] = 0.5
rr, cc = disk((128, 80), 20)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom[rr[valid], cc[valid]] = 1.0
rr, cc = disk((128, 170), 25)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom[rr[valid], cc[valid]] = 0.8
rr, cc = disk((90, 128), 10)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom[rr[valid], cc[valid]] = 0.9

# Compute projections at specific angles
angles = [0, 45, 90, 135]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, angle in enumerate(angles):
    # Compute Radon Transform at this angle
    theta = np.array([angle])
    projection = radon(phantom, theta=theta, circle=True)
    
    # Show the phantom with ray direction
    axes[0, i].imshow(phantom, cmap='gray', vmin=0, vmax=1)
    # Draw arrows showing ray direction
    rad = np.radians(angle)
    for pos in range(40, 220, 40):
        dx = -np.sin(rad) * 30
        dy = np.cos(rad) * 30
        start_x = pos + np.cos(rad) * 0
        start_y = 128 - np.sin(rad) * (pos - 128)
        axes[0, i].annotate('', xy=(start_x + dx, start_y + dy),
                           xytext=(start_x - dx, start_y - dy),
                           arrowprops=dict(arrowstyle='->', color='red', lw=1))
    axes[0, i].set_title(f'Rays at $\\theta$ = {angle}°', fontsize=11)
    
    # Show the projection
    axes[1, i].plot(projection, 'b-', linewidth=2)
    axes[1, i].set_title(f'Projection g($\\rho$, {angle}°)', fontsize=11)
    axes[1, i].set_xlabel('$\\rho$ (detector position)')
    axes[1, i].set_ylabel('Line integral')
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Parallel-Beam Projections at Different Angles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Properties of the Radon Transform

### Linearity
$$\mathcal{R}\{\alpha f + \beta g\} = \alpha \, \mathcal{R}\{f\} + \beta \, \mathcal{R}\{g\}$$

### Shifting Property
If $f(x, y)$ is shifted to $f(x - x_0, y - y_0)$, then:
$$g(\rho, \theta) \rightarrow g(\rho - x_0\cos\theta - y_0\sin\theta, \theta)$$

A shift in the spatial domain causes a **sinusoidal shift** in the $\rho$ direction — this is why the collection of projections is called a **sinogram**!

### Rotation Property
If $f(x, y)$ is rotated by angle $\theta_0$, then:
$$g(\rho, \theta) \rightarrow g(\rho, \theta - \theta_0)$$

Rotation in the spatial domain = shift in the angular coordinate.

In [ ]:
# Demonstrate why it's called a "sinogram"
# A point source traces a sinusoidal path in the Radon domain
size = 256
theta = np.linspace(0., 180., 180, endpoint=False)

# Create images with single point sources at different locations
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Point at center
img1 = np.zeros((size, size))
rr, cc = disk((128, 128), 5)
img1[rr, cc] = 1.0
sinogram1 = radon(img1, theta=theta, circle=True)

# Point off-center
img2 = np.zeros((size, size))
rr, cc = disk((80, 160), 5)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
img2[rr[valid], cc[valid]] = 1.0
sinogram2 = radon(img2, theta=theta, circle=True)

# Two points
img3 = np.zeros((size, size))
rr, cc = disk((100, 80), 5)
img3[rr, cc] = 1.0
rr, cc = disk((160, 180), 5)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
img3[rr[valid], cc[valid]] = 1.0
sinogram3 = radon(img3, theta=theta, circle=True)

images = [img1, img2, img3]
sinograms = [sinogram1, sinogram2, sinogram3]
titles = ['Point at Center', 'Point Off-Center', 'Two Points']

for i in range(3):
    axes[0, i].imshow(images[i], cmap='gray')
    axes[0, i].set_title(titles[i], fontsize=12)
    
    axes[1, i].imshow(sinograms[i], cmap='hot', aspect='auto',
                     extent=[theta[0], theta[-1], sinograms[i].shape[0], 0])
    axes[1, i].set_title(f'Sinogram of "{titles[i]}"', fontsize=12)
    axes[1, i].set_xlabel('$\\theta$ (degrees)')
    axes[1, i].set_ylabel('$\\rho$ (detector position)')

plt.suptitle('Why "Sinogram"? — A Point Traces a Sinusoidal Curve', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("A point at the center produces a horizontal line in the sinogram (rho = 0 for all angles).")
print("A point off-center traces a sinusoidal curve — hence the name 'sinogram'!")
print("Two points produce two overlapping sinusoidal curves.")

## 6. A Simple Example: Radon Transform of Basic Shapes

Let's compute the Radon Transform of simple geometric objects to build intuition.

In [ ]:
# Radon Transform of basic shapes
size = 256
theta = np.linspace(0., 180., 180, endpoint=False)

# Shape 1: Circle (symmetric)
circle = np.zeros((size, size))
rr, cc = disk((128, 128), 50)
circle[rr, cc] = 1.0

# Shape 2: Rectangle
rect = np.zeros((size, size))
rect[88:168, 78:178] = 1.0

# Shape 3: Ellipse
ellip = np.zeros((size, size))
rr, cc = ellipse(128, 128, 40, 80)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ellip[rr[valid], cc[valid]] = 1.0

shapes = [circle, rect, ellip]
shape_names = ['Circle', 'Rectangle', 'Ellipse']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, (shape, name) in enumerate(zip(shapes, shape_names)):
    sinogram = radon(shape, theta=theta, circle=True)
    
    axes[0, i].imshow(shape, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'{name}', fontsize=12)
    
    axes[1, i].imshow(sinogram, cmap='hot', aspect='auto',
                     extent=[theta[0], theta[-1], sinogram.shape[0], 0])
    axes[1, i].set_title(f'Sinogram of {name}', fontsize=12)
    axes[1, i].set_xlabel('$\\theta$ (degrees)')
    axes[1, i].set_ylabel('$\\rho$')
    axes[1, i].colorbar = plt.colorbar(axes[1, i].images[0], ax=axes[1, i])

plt.suptitle('Radon Transform of Basic Shapes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Circle: The sinogram is nearly constant across angles (rotational symmetry).")
print("Rectangle: The sinogram varies with angle, widest along diagonals.")
print("Ellipse: The sinogram shows smooth variation reflecting the elliptical shape.")

## 7. The Shepp-Logan Phantom

The **Shepp-Logan phantom** is a standard test image in CT reconstruction. It was designed by Larry Shepp and Benjamin Logan in 1974 to simulate a human head cross-section.

It consists of overlapping ellipses with different intensities, representing:
- Skull (outer ellipse)
- Brain tissue (inner regions)
- Ventricles and tumors (small features)

In [ ]:
from skimage.data import shepp_logan_phantom

# Create the Shepp-Logan phantom
phantom = shepp_logan_phantom()

# Compute its Radon Transform (sinogram)
theta = np.linspace(0., 180., 360, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Shepp-Logan Phantom (400×400)', fontsize=12)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

im = axes[1].imshow(sinogram, cmap='hot', aspect='auto',
                    extent=[theta[0], theta[-1], sinogram.shape[0], 0])
axes[1].set_title('Sinogram (Radon Transform)', fontsize=12)
axes[1].set_xlabel('Projection Angle $\\theta$ (degrees)')
axes[1].set_ylabel('Detector Position $\\rho$')
plt.colorbar(im, ax=axes[1], label='Projection Value')

plt.suptitle('Shepp-Logan Phantom and Its Radon Transform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Phantom shape: {phantom.shape}")
print(f"Sinogram shape: {sinogram.shape} (detector_positions × angles)")
print(f"Number of projection angles: {len(theta)}")

## Summary

What we learned:
1. The **Radon Transform** computes line integrals of a function along all possible rays
2. In CT, X-ray attenuation follows **Beer-Lambert's law**, and the measured projections are line integrals
3. A projection at angle $\theta$ is: $g(\rho, \theta) = \int\int f(x,y) \, \delta(x\cos\theta + y\sin\theta - \rho) \, dx \, dy$
4. The collection of all projections is called a **sinogram** (because a point traces a sinusoidal curve)
5. The Radon Transform is **linear**, and spatial shifts produce sinusoidal shifts in the sinogram
6. The **Shepp-Logan phantom** is the standard test image for CT reconstruction algorithms